In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

google_maps_api_key = os.getenv('GOOGLE_MAPS_API_KEY')

if google_maps_api_key:
    print("Google Maps API Key loaded successfully.")
else:
    print("Google Maps API Key is not set.")

Google Maps API Key loaded successfully.


In [ ]:
import requests
import pandas as pd
import random

city_data = {
    "leeds": {
        "districts": [
            "LS1", "LS2", "LS3", "LS4", "LS5", "LS6", "LS7", "LS8", "LS9", "LS10",
            "LS11", "LS12", "LS13", "LS14", "LS15", "LS16", "LS17", "LS18", "LS19",
            "LS20", "LS21", "LS22", "LS23", "LS24", "LS25", "LS26", "LS27", "LS28", "LS29"
        ],
        "center": (53.8008, -1.5491)  # Leeds Station / City Square
    },
    "bristol": {
        "districts": ["BS1", "BS2", "BS3", "BS4", "BS5", "BS6", "BS7", "BS8", "BS9"],
        "center": (51.4545, -2.5879)  # Bristol city center
    },
    "sheffield": {
        "districts": ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
        "center": (53.3811, -1.4701)  # Sheffield city center
    },
    "manchester": {
        "districts": ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9"],
        "center": (53.4808, -2.2426)  # Manchester Piccadilly area
    },
    "bradford": {
        "districts": ["BD1", "BD2"],
        "center": (53.7913, -1.7494)  # Bradford city center
    },
    "london": {
        "districts": ["N1", "N2", "NW1", "SE1", "SW1", "E1", "W1", "EC1", "WC1"],
        "center": (51.5074, -0.1278)  # Charing Cross, Central London
    }
}

def fetch_sample_postcodes(district, n_samples=10):
    """Fetch random sample of postcodes for a given district prefix."""
    url = f"https://api.postcodes.io/postcodes?q={district}"
    r = requests.get(url)
    if r.status_code != 200:
        return []
    results = r.json().get("result", [])
    all_postcodes = [r["postcode"] for r in results if r["postcode"].startswith(district)]
    return random.sample(all_postcodes, min(n_samples, len(all_postcodes)))

# Step 2: Sample equally across districts
city = "bradford"
districts = city_postcode_districts[city]
per_district_sample = 3

sampled_postcodes = []
for d in districts:
    sampled_postcodes.extend(fetch_sample_postcodes(d, per_district_sample))

# Step 3: Get lat/lon in bulk
def bulk_geocode(postcodes):
    url = "https://api.postcodes.io/postcodes"
    headers = {"Content-Type": "application/json"}
    payload = {"postcodes": postcodes}
    r = requests.post(url, json=payload, headers=headers)
    results = r.json()["result"]
    return [
        {
            "postcode": res["query"],
            "lat": res["result"]["latitude"] if res["result"] else None,
            "lon": res["result"]["longitude"] if res["result"] else None
        }
        for res in results
    ]

# Split into batches of 100
batches = [sampled_postcodes[i:i+100] for i in range(0, len(sampled_postcodes), 100)]
geo_data = []
for batch in batches:
    geo_data.extend(bulk_geocode(batch))

df = pd.DataFrame(geo_data).dropna()
print(df.head())


  postcode        lat       lon
0  BD1 1BL  53.794163 -1.752492
1  BD1 1AF  53.794994 -1.750301
2  BD1 1EZ  53.793603 -1.751190
3  BD2 1AA  53.809632 -1.757427
4  BD2 1AP  53.811485 -1.758298


In [ ]:
import time

GOOGLE_API_KEY = "YOUR_API_KEY"  # Replace with your actual key
ORIGIN = "Leeds Station, Leeds"  # or use (lat, lon) as string: "53.7950,-1.5486"
MODE = "transit"  # change to "bicycling" for cycling mode

def get_travel_time(origin, dest_lat, dest_lon, mode="transit"):
    dest = f"{dest_lat},{dest_lon}"
    url = (
        f"https://maps.googleapis.com/maps/api/distancematrix/json"
        f"?origins={origin}"
        f"&destinations={dest}"
        f"&mode={mode}"
        f"&key={GOOGLE_API_KEY}"
    )
    r = requests.get(url)
    if r.status_code != 200:
        return None
    result = r.json()
    try:
        duration_sec = result["rows"][0]["elements"][0]["duration"]["value"]
        return duration_sec / 60  # convert to minutes
    except (KeyError, IndexError, TypeError):
        return None

# Add travel times
df["travel_time_min"] = df.apply(
    lambda row: get_travel_time(ORIGIN, row["lat"], row["lon"], mode=MODE),
    axis=1
)

# Optional: Save to file
df.to_csv("leeds_postcodes_with_travel_times.csv", index=False)


In [4]:
import folium
from folium.plugins import MarkerCluster

def plot_postcodes_on_map(postcodes_df, center=None, zoom=11, color='blue'):
    """
    Plot a set of postcodes (lat/lon) on an interactive folium map.

    Args:
        postcodes_df (pd.DataFrame): Must contain 'lat' and 'lon' columns.
        center (tuple): Optional (lat, lon) to center the map.
        zoom (int): Initial zoom level.
        color (str): Marker color.

    Returns:
        folium.Map object.
    """
    if center is None:
        center = (
            postcodes_df["lat"].mean(),
            postcodes_df["lon"].mean()
        )

    m = folium.Map(location=center, zoom_start=zoom, tiles="CartoDB positron")

    marker_cluster = MarkerCluster().add_to(m)

    for _, row in postcodes_df.iterrows():
        popup_text = row.get("postcode", "Unknown")
        folium.CircleMarker(
            location=(row["lat"], row["lon"]),
            radius=4,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.6,
            popup=popup_text
        ).add_to(marker_cluster)

    return m


In [14]:
#plot_postcodes_on_map(df)

print(len(df))


6


In [ ]:
import googlemaps
from datetime import datetime
import random

gmaps = googlemaps.Client(key=google_maps_api_key)

coords = get_coords((51.507200, -0.127600), num=10)


In [21]:
def pick_city(city='London'):
    
    if city=='London':
        centre_point = (51.507200, -0.127600)
        
    return centre_point


def get_coords(centre, num=100, max_dist=0.1):
    
    x, y = centre
    coords = []
    
    for _ in range(num):
        new_x = x + random.uniform(-max_dist, max_dist)
        new_y = y + random.uniform(-max_dist, max_dist)
        
        coords.append((new_x, new_y))
    
    return coords


In [ ]:
coords = get_coords((51.507200, -0.127600), num=10)

for coord in coords:
    print(coord)


[(51.591737483540015, -0.03219684172802301), (51.4157418621727, -0.2157265775426122), (51.52421426627896, -0.20473115096749872), (51.449195043803805, -0.1042450901917077), (51.55265901746759, -0.2190822540555757), (51.43355856877363, -0.08960303149171644), (51.52680157494993, -0.06067199708004009), (51.56049467803123, -0.07565877445483357), (51.557805647328934, -0.04260285403272665), (51.579156952993365, -0.1982601058067141)]


In [ ]:
load_api_key()

pick_city():
    '''dict to match a city to a '''

generate_coords():
    '''100 randomly placed coords, based on a certain distance from a city's centre point'''

set_start_loc()

for i in size coords_list:
    get_travel_time()

plot((x,y)=coords_list, color=travel_time_inverse, smoothing='gaussian'):
    '''can trim the dispaly to remove edges'''

https://docs.mapbox.com/api/navigation/isochrone/
